# Notebook 22 — HTML parsing & visible text (`html.parser`)

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Intermediate → Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `21-venv-and-dependency-pins.ipynb`

**Next up:** `../python-foundations-beginner-to-advanced.ipynb` + `../CURRICULUM-A-Z.md`

---

## Learning objectives

- Prefer parsers over naive regex for markup-heavy retrieval sources.
- Skip **`script` / `style` / `noscript`** islands safely.
- Produce coarse plain text suitable for chunking previews.

## Table of contents

1. **`HTMLParser` mechanics**
2. **Skipping hostile subtrees**
3. **Whitespace normalization**
4. **Progressive drills — tiny fragment → nested tags → entities**
5. **Exercise — `visible_html_text`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · `HTMLParser` mechanics

*Explanation:* Streaming callbacks (**`handle_starttag`**, **`handle_data`**) mirror SAX pipelines — constant memory vs **`BeautifulSoup`** (fine later).


In [ ]:
from html.parser import HTMLParser


class EchoParser(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.chunks: list[str] = []

    def handle_data(self, data: str) -> None:
        text = data.strip()
        if text:
            self.chunks.append(text)


parser = EchoParser()
parser.feed("<p>hello</p><p>world</p>")
print(parser.chunks)


## 2 · Skipping hostile subtrees

*Explanation:* Retrieval ignores **`script`** noise — gate **`handle_data`** when inside forbidden tags.


In [ ]:
from html.parser import HTMLParser


class VisibleParser(HTMLParser):
    _blocked = frozenset({"script", "style", "noscript"})

    def __init__(self) -> None:
        super().__init__()
        self.depth = 0
        self.parts: list[str] = []

    def handle_starttag(self, tag: str, attrs) -> None:
        if tag.lower() in self._blocked:
            self.depth += 1

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() in self._blocked and self.depth:
            self.depth -= 1

    def handle_data(self, data: str) -> None:
        if self.depth:
            return
        chunk = data.strip()
        if chunk:
            self.parts.append(chunk)


html = "<div>Keep<script>secret()</script><p>visible</p></div>"
vp = VisibleParser()
vp.feed(html)
print(vp.parts)


## 3 · Entities & normalization

*Explanation:* **`html.unescape`** repairs **`&amp;`** before hashing chunks.


In [ ]:
import html

blob = html.unescape("Latency &amp; tokens")
print(blob)


---

## Progressive drills — **easy → harder**

**Scraped docs** powering RAG benefit from deterministic parsers — drills mimic stripping boilerplate chrome.


### A · Easiest — ignore `<nav>` chrome

Treat navigation buckets like **`script`** when prototyping.


In [ ]:
from html.parser import HTMLParser


class NavSkip(HTMLParser):
    def __init__(self) -> None:
        super().__init__()
        self.skip_depth = 0
        self.words: list[str] = []

    def handle_starttag(self, tag: str, attrs) -> None:
        if tag.lower() == "nav":
            self.skip_depth += 1

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() == "nav" and self.skip_depth:
            self.skip_depth -= 1

    def handle_data(self, data: str) -> None:
        if self.skip_depth:
            return
        token = data.strip()
        if token:
            self.words.append(token)


sample = "<nav>Skip me</nav><article>Keep me</article>"
parser = NavSkip()
parser.feed(sample)
print(parser.words)


### B · Medium — flatten doubled whitespace

Collapse interiors before embeddings comparisons.


In [ ]:
import re

messy = "hello    rag\nagent"
print(re.sub(r"\s+", " ", messy.strip()))


### C · Harder — guard unknown tags

Lower-case comparisons defend against **`DIV`** variance.


In [ ]:
print("DIV".lower() == "div")


### Exercise — `visible_html_text`

Implement **`visible_html_text(fragment: str) -> str`** using an **`HTMLParser`** subclass like **`VisibleParser`** (skip **`script`**, **`style`**, **`noscript`**). Concatenate surviving stripped text chunks with a single space (`" ".join(parts)`), then strip.

Must satisfy **`visible_html_text("<p>a</p><script>x()</script><p>b</p>") == "a b"`**.


In [ ]:
def visible_html_text(fragment: str) -> str:
    raise NotImplementedError


assert visible_html_text("<p>a</p><script>x()</script><p>b</p>") == "a b"
print("OK")


### Solution — visible_html_text

<details>
<summary>Click to expand</summary>

```python
from html.parser import HTMLParser


class VisibleParser(HTMLParser):
    _blocked = frozenset({"script", "style", "noscript"})

    def __init__(self) -> None:
        super().__init__()
        self.depth = 0
        self.parts: list[str] = []

    def handle_starttag(self, tag: str, attrs) -> None:
        if tag.lower() in self._blocked:
            self.depth += 1

    def handle_endtag(self, tag: str) -> None:
        if tag.lower() in self._blocked and self.depth:
            self.depth -= 1

    def handle_data(self, data: str) -> None:
        if self.depth:
            return
        chunk = data.strip()
        if chunk:
            self.parts.append(chunk)


def visible_html_text(fragment: str) -> str:
    parser = VisibleParser()
    parser.feed(fragment)
    return " ".join(parser.parts).strip()
```

</details>
